In [1]:
import faiss
from sentence_transformers import SentenceTransformer
import os

model = SentenceTransformer("all-MiniLM-L6-v2")


/Users/vitolin/Desktop/code/github/vectorsearch/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load text from a file
texts = None
with open('english-5k.csv', 'r', encoding='utf-8') as f:
    texts = [line.strip() for line in f]


In [3]:
vector_db_path = "english_5k_hnsw.index"


In [4]:
if not os.path.exists(vector_db_path): # skip creating the db if it already exists
    embeddings = model.encode(texts, convert_to_numpy=True)
    embeddings = embeddings.astype("float32")
    print("embeddings.shape:", embeddings.shape)

    d = embeddings.shape[1]
    M = 32
    index = faiss.IndexHNSWFlat(d, M)
    index.add(embeddings)
    print("Total vectors in index:", index.ntotal)

    # save the index
    faiss.write_index(index, vector_db_path)


In [5]:
# load the index
index = faiss.read_index(vector_db_path)


In [6]:
query = "king"
query_embedding = model.encode([query], convert_to_numpy=True)
query_embedding = query_embedding.astype("float32")

k = 5  # number of nearest neighbors to retrieve
D, I = index.search(query_embedding, k)
print(f"Top {k} nearest neighbors for '{query}':")
for dist, idx in zip(D[0], I[0]):
    print(f"Word: {texts[idx]}, Index: {idx}, Distance: {dist}")

# try to get queen with adding vectors together
q1 = "king"
q2 = "male"
q3 = "female"

q1_emb = model.encode([q1], convert_to_numpy=True).astype("float32")
q2_emb = model.encode([q2], convert_to_numpy=True).astype("float32")
q3_emb = model.encode([q3], convert_to_numpy=True).astype("float32")

combined_emb = q1_emb - q2_emb + q3_emb
D, I = index.search(combined_emb, k)
print(f"\nTop {k} nearest neighbors for combination of '{q1}' - '{q2}' + '{q3}':")
for dist, idx in zip(D[0], I[0]):
    print(f"Word: {texts[idx]}, Index: {idx}, Distance: {dist}")


Top 5 nearest neighbors for 'king':
Word: king, Index: 2082, Distance: 9.338125975602574e-13
Word: kingdom, Index: 3948, Distance: 0.5331282615661621
Word: queen, Index: 3616, Distance: 0.6385747194290161
Word: royal, Index: 3786, Distance: 0.7182959318161011
Word: champion, Index: 2441, Distance: 0.860819935798645

Top 5 nearest neighbors for combination of 'king' - 'male' + 'female':
Word: king, Index: 2082, Distance: 0.532413899898529
Word: queen, Index: 3616, Distance: 0.7292802333831787
Word: kingdom, Index: 3948, Distance: 0.9252558350563049
Word: royal, Index: 3786, Distance: 1.064408779144287
Word: champion, Index: 2441, Distance: 1.196191668510437


In [7]:
# applying this to more words
m = "male"
f = "female"

m_emb = model.encode([m], convert_to_numpy=True).astype("float32")
f_emb = model.encode([f], convert_to_numpy=True).astype("float32")

print(f"\nTop nearest neighbor for combination of each word - '{m}' + '{f}' that is not the word:")
print("word,nearest_neighbor,distance")
for word in texts[:100]:
    w_emb = model.encode([word], convert_to_numpy=True).astype("float32")
    combined_emb = w_emb - m_emb + f_emb
    D, I = index.search(combined_emb, 2)
    for dist, idx in zip(D[0], I[0]):
        if texts[idx] != word:
            print(f"{word},{texts[idx]},{dist}")



Top nearest neighbor for combination of each word - 'male' + 'female' that is not the word:
word,nearest_neighbor,distance
the,her,0.9359264373779297
be,being,1.2402640581130981
and,also,1.0675610303878784
of,for,1.326987862586975
a,woman,1.1341049671173096
in,into,1.055591106414795
to,her,1.1134130954742432
have,with,1.5418050289154053
it,its,0.9108779430389404
I,she,0.8766992092132568
that,those,1.1553581953048706
for,of,1.2625893354415894
you,she,0.9023616909980774
he,she,0.4134378135204315
with,by,1.301674485206604
on,upon,1.343461275100708
do,go,1.3980592489242554
say,let,1.3741874694824219
this,she,1.0608851909637451
they,them,0.7060208916664124
at,her,1.3411551713943481
but,however,0.880733847618103
we,our,0.9246466755867004
his,her,0.43444371223449707
from,originally,1.213111400604248
not,no,1.1045726537704468
n't,she,1.353298544883728
by,with,1.0994391441345215
she,her,0.6783744096755981
or,and/or,1.0398449897766113
as,she,1.3591182231903076
what,her,1.1476038694381714
go,sta